# Look at the `sleep_apnea` database

A small notebook to check the database and look at some data.

**Before:** start PostgreSQL from the project root:
```
docker compose -f db/docker-compose.yml up -d
```
Optional, to have one row in every table: `python db/viewer/seed_demo_rows.py`

Run the cells from top to bottom (Shift + Enter).

## 1. Connect

In [ ]:
import pandas as pd
import psycopg
import matplotlib.pyplot as plt

conn = psycopg.connect(host="localhost", port=5432, dbname="sleep_apnea",
                       user="postgres", password="admin", autocommit=True)

def sql(query, params=None):
    """Run a query and return the result as a table."""
    with conn.cursor() as cur:
        cur.execute(query, params)
        return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])

print("connected")

## 2. All tables and how many rows they have

In [ ]:
tables = sql("""SELECT table_name FROM information_schema.tables
                 WHERE table_schema = 'public' ORDER BY table_name""")["table_name"]

counts = pd.DataFrame({"table": tables,
                       "rows": [sql(f'SELECT count(*) AS n FROM "{t}"')["n"][0] for t in tables]})
counts

## 3. The 7 service users

In [ ]:
sql("""SELECT rolname AS user FROM pg_roles
       WHERE rolname IN ('patient_seed', 'apnea_detector', 'ahi_triage', 'model_trainer',
                         'alert_service', 'doctor_dashboard', 'carbon_scheduler')
       ORDER BY rolname""")

## 4. The demo patients

In [ ]:
sql("SELECT * FROM patients ORDER BY patient_id")

## 5. Look at any table

Change `TABLE` to one of the names from step 2. The signal columns of `raw_nights` are long, so they are left out here.

In [ ]:
TABLE = "results"

if TABLE == "raw_nights":
    display(sql("SELECT night_id, patient_id, source, recorded_at FROM raw_nights"))
else:
    display(sql(f'SELECT * FROM "{TABLE}" LIMIT 20'))

## 6. Plot the SpO₂ of one night

Takes the first night in `raw_nights`. If the table is still empty, it reads the real night from the data file instead.

In [ ]:
import numpy as np
from pathlib import Path

nights = sql("SELECT night_id FROM raw_nights ORDER BY night_id")

if len(nights):
    night = nights["night_id"][0]
    spo2 = sql("SELECT spo2 FROM raw_nights WHERE night_id = %s", (night,))["spo2"][0]
    spo2 = np.array([np.nan if v is None else v for v in spo2], dtype=float)
    where = "database"
else:
    night = "ucddb025"
    spo2 = np.load(Path("..") / "data" / "clean" / "signals" / f"{night}.npz")["spo2"]
    where = "data file (raw_nights is empty)"

minutes = np.arange(len(spo2)) / 60          # SpO2 is 1 value per second

plt.figure(figsize=(12, 3.5))
plt.plot(minutes, spo2, linewidth=0.6)
plt.axhline(80, color="red", linestyle="--", label="alert threshold 80 %")
plt.title(f"SpO₂ – {night}  (from {where})")
plt.xlabel("minute of the night")
plt.ylabel("SpO₂ %")
plt.legend()
plt.show()

print(f"lowest SpO₂: {np.nanmin(spo2):.0f} %,  seconds below 80 %: {(spo2 < 80).sum()}")

## 7. Alerts of that night

In [ ]:
sql("SELECT minute, spo2_min, detected_at, stored_at FROM alerts WHERE night_id = %s ORDER BY minute", (night,))

## 8. Close the connection

In [ ]:
conn.close()